# Dashboards, Alerts, and SLOs for Agent Systems
Companion notebook for the **Dashboards, Alerts, and SLOs for Agent Systems** section of Chapter 9.

A production dashboard tracks four panels together: **quality**, **latency**, **cost**, and **reliability**. Alerts should fire on important, sustained shifts -- not every small wobble -- using rolling windows to reduce noise.

The book's alert definitions, as real Python conditions instead of loose YAML:

In [1]:
alerts = [
    {
        "name": "task_success_drop",
        "condition": lambda m: m["success_rate_1h"] < 0.82 and m["runs_1h"] > 200,
        "severity": "high",
    },
    {
        "name": "latency_regression",
        "condition": lambda m: m["p95_latency_15m"] > 12,  # seconds
        "severity": "medium",
    },
    {
        "name": "cost_spike",
        "condition": lambda m: m["cost_per_success_1h"] > 1.5 * m["baseline_cost_per_success"],
        "severity": "medium",
    },
]

In [2]:
def evaluate_alerts(metrics: dict, alert_defs: list[dict]) -> list[dict]:
    fired = []
    for alert in alert_defs:
        if alert["condition"](metrics):
            fired.append({"name": alert["name"], "severity": alert["severity"]})
    return fired


# A normal hour: nothing should fire.
healthy_metrics = {
    "success_rate_1h": 0.91, "runs_1h": 480,
    "p95_latency_15m": 6.2,
    "cost_per_success_1h": 0.05, "baseline_cost_per_success": 0.045,
}
print("healthy window:", evaluate_alerts(healthy_metrics, alerts))

# After a bad prompt rollout: success drops, latency and cost both rise.
degraded_metrics = {
    "success_rate_1h": 0.74, "runs_1h": 350,
    "p95_latency_15m": 13.8,
    "cost_per_success_1h": 0.09, "baseline_cost_per_success": 0.045,
}
print("degraded window:", evaluate_alerts(degraded_metrics, alerts))

healthy window: []
degraded window: [{'name': 'task_success_drop', 'severity': 'high'}, {'name': 'latency_regression', 'severity': 'medium'}, {'name': 'cost_spike', 'severity': 'medium'}]


A dashboard's four panels, populated from the same kind of run data as notebook 02 -- context (which model, prompt, or release) is what makes a chart actionable:

In [3]:
def build_dashboard_snapshot(metrics: dict, breakdown: dict) -> dict:
    return {
        "quality": {"task_success_rate": metrics["success_rate_1h"]},
        "latency": {"p95_s": metrics["p95_latency_15m"]},
        "cost": {"cost_per_success_usd": metrics["cost_per_success_1h"]},
        "reliability": {"active_alerts": [a["name"] for a in evaluate_alerts(metrics, alerts)]},
        "breakdown": breakdown,
    }


snapshot = build_dashboard_snapshot(
    degraded_metrics,
    breakdown={"prompt_version": "refund_v13", "model": "gpt-4.1", "release_id": "rel_2026_04_28_01"},
)
import json
print(json.dumps(snapshot, indent=2))

{
  "quality": {
    "task_success_rate": 0.74
  },
  "latency": {
    "p95_s": 13.8
  },
  "cost": {
    "cost_per_success_usd": 0.09
  },
  "reliability": {
    "active_alerts": [
      "task_success_drop",
      "latency_regression",
      "cost_spike"
    ]
  },
  "breakdown": {
    "prompt_version": "refund_v13",
    "model": "gpt-4.1",
    "release_id": "rel_2026_04_28_01"
  }
}


**SLOs** should reflect what the user experiences, not raw uptime -- "95% of refund tasks complete within 20 seconds" beats "99.9% API uptime" because a user never sees the second number directly:

In [4]:
def slo_met(latencies_s: list[float], target_s: float, target_fraction: float = 0.95) -> bool:
    within_target = sum(1 for lat in latencies_s if lat <= target_s)
    return (within_target / len(latencies_s)) >= target_fraction


sample_latencies = [8.1, 9.4, 11.0, 7.8, 19.5, 10.2, 9.9, 8.8, 22.1, 9.1]
print("95% of refund tasks complete within 20s:", slo_met(sample_latencies, target_s=20.0))

95% of refund tasks complete within 20s: False
